# Stop location, recreation stops: multinomial logit over 1,588 zones, sequential anchors, location mixture (article table est_stop_RECREATION_stop)

A model of MAST (Maryland Activity Scheduling and Travel model), the activity-based model of the MILES platform, built on the ABIT code base.

## What this model answers

Where does a recreation stop on a tour go? The model compares the 1,588 zones of the model (full choice set) by detour, attraction, area and land use, with detour effects that differ by tour mode and by the direction of the stop.

| | |
|---|---|
| Choice | The zone of a recreation stop on a tour |
| Alternatives | The 1,588 zones of the model (full choice set) |
| Decision makers | Persons making a recreation stop |
| Data | Regional Travel Survey 2017/18 (Baltimore and Washington regions), estimation half |

## How to run

- Kernel: *Python (biogeme)* (`~/.venvs/biogeme`, Biogeme 3.3). Then *Run All*.
- `RUN_DATA_PREP`: rebuild the records and tables in a scratch copy and compare them with the shipped ones.
- `RUN_CHECK`: log-likelihood at the reference coefficients, no estimation (about 32 min).
- `RUN_ESTIMATION`: run the Biogeme estimation (off by default).
- `RUN_HOLDOUT`: log-likelihood of the hold-out half (as long as the check; off by default).

## Contents

This notebook goes from the survey records to the estimates of this model:

1. Raw data: survey files, estimation and hold-out halves, the records of this model
2. Data preparation: filters from the survey to the records (records kept and dropped), rebuild check against the shipped tables
3. Choice set and estimation data
4. Descriptive statistics and figures; 4.1 every explanatory variable of the model in the data
5. Model specification (the folder's `model/estimate_stop_recreation.py`, unchanged)
6. Correlation of the explanatory variables (heatmaps, VIF)
7. Log-likelihood at the reference coefficients
8. Biogeme estimation
9. Results: article format, what the coefficients say, fit, comparison with the article, correlation of the estimates
10. Hold-out sample
11. Glossary

## Model at a glance

**1. Who decides.** Persons making a recreation stop. Data: Regional Travel Survey 2017/18 (Baltimore and Washington regions), estimation half; 5,064 stops.

**2. What is chosen (dependent variable).** The zone of a recreation stop on a tour. Alternatives: The 1,588 zones of the model (full choice set).

**3. Explanatory variables** (signs of the article estimates; terms specific to an alternative or purpose are listed per series):

| Group | Variable | Sign |
|---|---|---|
| Detour and impedance | detour 1-2 km | − |
| Detour and impedance | detour 2-5 km | − |
| Detour and impedance | detour 5-10 km | − |
| Detour and impedance | detour 10-20 km | − |
| Detour and impedance | detour > 20 km | − |
| Detour and impedance | mode-choice logsum, detour | + |
| Anchor zones | stop in the zone of the previous location | + |
| Anchor zones | stop in the zone of the next anchor | + |
| Area | Baltimore City | − |
| Area | Washington DC | + |
| Area | high job-density zone | + |
| Area | Virginia | + |
| Area | Delaware, Pennsylvania, West Virginia | + |
| Land use | log(zone area, km2) | + |
| Land use | land-use mix (entropy) | + |
| Boundaries | outside the counties of both anchors | − |
| Boundaries | outside the states of both anchors | − |
| Interactions with log(1 + detour) | log(1 + detour) x tour mode not car | − |
| Interactions with log(1 + detour) | log(1 + detour) x inbound stop | + |
| Attraction | log(1 + retail jobs) | + |

**4. How person and household traits enter.** No person or household traits enter; the tour mode and the direction of the stop (inbound) enter as interactions with log detour.

**5. How to read a coefficient.** Positive makes the alternative more likely, negative less likely; exp(coefficient) is the factor by which the odds change for one unit more.

In [ ]:
# switches (see How to run) and set-up: imports, folders, helper modules
RUN_DATA_PREP  = True    # rebuild the records and estimation tables in a scratch copy and compare with the shipped tables (about 4 min)
RUN_CHECK      = True    # log-likelihood at the reference coefficients  (this model: about 32 min)
RUN_ESTIMATION = False   # Biogeme estimation, written to output/ (False: results are read from output/ if there are any)
RUN_HOLDOUT    = False    # log-likelihood of the hold-out half at the reference coefficients  (this model: as long as the check)

import os, sys, json, glob, inspect, importlib, time, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt; import nb_reader as rd; import figs as fg; import abit_figs as af
from IPython.display import Markdown
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "stop_recreation"; MODEL_FOLDER = "03_stop_recreation"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", os.path.basename(FOLDER))

## 1. Raw data

*Where the records come from. Look for: the files and the record counts.*

### 1.1 The 2017/18 Regional Travel Survey

All MAST models are estimated on the 2017/18 RTS of the Baltimore and Washington regions (weekday travel day). Three files
are shipped in `../00_shared_data/rts_2017_18/raw`:

- `RTS_raw_flat_data.csv.gz`: the survey as received, one row per trip with its household, person and vehicle fields;
- `rts_trips_clean.csv.gz`: the trips after the survey cleaning (one row per trip, departure / arrival in minutes);
- `rts_tours.csv.gz`: home-based tours, work-based subtours and open tours built from the clean trips.

The cleaning script of the last two files is not shipped; the cleaned files are the starting point of the preparation here.
Race and ethnicity fields are not read.

In [ ]:
# 1.1 The 2017/18 Regional Travel Survey
flat, trips, tours = nt.rts_raw()
split = pd.read_csv(os.path.join(nt.ABIT_SHARED, "rts_2017_18", "split", "rts_split.csv"))
display(nt.rts_counts(flat, trips, tours, split))
show = ["tripid", "household_id", "person_id", "tripno", "o_activity", "d_activity", "dep_min", "arr_min", "travel_mode",
        "distance", "hhsize", "numvehicle", "hh_income_broad", "age", "employment_status", "student_status", "wttrdfin"]
trips[[c for c in show if c in trips]].head()

### 1.2 Estimation and hold-out halves

The households are split in two halves (`make_rts_split.py`): strata of home county x household size (1 to 5+), a seeded
shuffle inside each stratum (seed 2026), then alternate assignment. Every model is estimated on the estimation half; the
hold-out (validation) half is the other half of the same survey and is not read by the estimation. With `RUN_DATA_PREP` the
split is rebuilt into a scratch folder and compared with the shipped `rts_split.csv`.

In [ ]:
# 1.2 Estimation and hold-out halves
REBUILD = os.path.join(OUT, "rebuild_check")            # scratch folder: provided files are never overwritten
if RUN_DATA_PREP:
    rc = nt.run_quiet(f"{sys.executable} make_rts_split.py", cwd=nt.ABIT_PREP, env={"ABIT_REBUILT": REBUILD})
    display(nt.compare_frames(pd.read_csv(os.path.join(REBUILD, "rts_2017_18", "split", "rts_split.csv")), split,
                              key=["household_id"], name="rts_split.csv"))
else:
    print("skipped (RUN_DATA_PREP = False)")
hh = flat.drop_duplicates("household_id").set_index("household_id")
hh["half"] = split.set_index("household_id")["half"]
t = pd.crosstab(hh["hhsize"].clip(upper=5).rename("household size (5 = 5+)"), hh["half"])
t.loc["all"] = t.sum()
t

### 1.3 Records of this model

The preparation turns the survey into the record files of `input/data/raw` (`dc_rec_<half>_taz.parquet`, `dc_cand_<half>_taz.parquet`, `dc_cfg_<half>_taz.parquet`):
one record per trip to a discretionary activity (shopping, recreation, accompanying, other) of a home-based tour:
either the primary activity of the tour (`kind = primary`, model 01) or an intermediate stop on a tour of any purpose
(`kind = stop`, models 02-05). Each record keeps its possible destination zones (`dc_cand`, tract or traffic zone of the
survey spread over model zones) and its possible anchor zones (`dc_cfg`).

In [ ]:
# 1.3 Records of this model
CODE = os.path.join(FOLDER, "input", "code"); RAW = os.path.join(FOLDER, "input", "data", "raw")
HALVES = ["estimation", "validation"]
raw_rec = {h: pd.read_parquet(os.path.join(RAW, f"dc_rec_{h}_taz.parquet")) for h in HALVES}
raw_cand = {h: pd.read_parquet(os.path.join(RAW, f"dc_cand_{h}_taz.parquet")) for h in HALVES}
by = ["kind", "purpose"] if "kind" in raw_rec["estimation"] else ["segment"]
t = pd.concat({h: raw_rec[h].groupby(by).size() for h in HALVES}, axis=1).fillna(0).astype(int)
t.loc[("all",) * len(by) if len(by) > 1 else "all", :] = t.sum()
display(t)
cols = ["tripid", "person_id", "kind", "purpose", "segment", "type", "tour_purpose", "position", "geoid", "km_rep", "tod",
        "age", "occ", "hhsize", "autos", "inc_cls", "home_idx", "home_zone_idx", "work_idx", "n_dz", "n_cfg"]
raw_rec["estimation"][[c for c in cols if c in raw_rec["estimation"]]].head()

## 2. Data preparation

*From the raw records to the estimation sample. Look for: the records kept and dropped at each filter. With RUN_DATA_PREP the rebuilt tables must equal the shipped ones.*

### 2.1 From survey trips to the records

`input/code/prep_dc_taz.py` (function `build`) works on the home-based tours of `rts_tours.csv.gz` and the trips of
`rts_trips_clean.csv.gz` of one half. The filters are mirrored below in pandas with the script's own helpers (survey loading,
split, survey population). This model keeps the stops with purpose RECREATION (on tours of any purpose).
The last step, locating every trip end in the model zones (survey traffic zone or tract spread over zones), needs the TAZ
crosswalks and runs only in the rebuild of section 2.2; its result is the shipped record file.

In [ ]:
# Data preparation
sys.path.insert(0, CODE)
import rts_common as RC, prep_dc_data as PD, dc_common as DC
S = RC.load_split()
tc = pd.read_csv(os.path.join(RC.RTS, "rts_trips_clean.csv.gz"), low_memory=False,
                 usecols=["tripid", "household_id", "person_id", "d_activity", "dep_min", "arr_min"])
tours_closed = RC.load_tours()            # anchor home or work (open tours dropped), purpose_abit of the primary activity
steps, N_RAW = {}, {}
KIND_STEP = lambda pr: ("tours with a shopping, recreation, accompany or other primary activity", len(pr))
for h in HALVES:
    P = PD.people(h)
    t = tc[tc.household_id.map(S).eq(h)].copy(); t["d_purpose"] = t.d_activity.map(RC.ACT_PURPOSE)
    s = [("tours in rts_tours.csv.gz (both halves)", len(tours))]
    tr = tours[tours.household_id.map(S).eq(h)]; s.append(("tours of the half", len(tr)))
    tr = tours_closed[tours_closed.household_id.map(S).eq(h) & tours_closed.anchor.eq("home")]; s.append(("home-based closed tours", len(tr)))
    tr = tr[~tr.primary_act.isin([1, -9])]; s.append(("primary activity known and not home", len(tr)))
    key = t.drop_duplicates(["person_id", "arr_min"]).set_index(["person_id", "arr_min"])
    hit = key.reindex(pd.MultiIndex.from_arrays([tr.person_id.values, tr.primary_arr.values]))
    tr = tr.assign(p_tripid=hit.tripid.values); tr = tr[tr.p_tripid.notna()]; s.append(("trip to the primary activity found", len(tr)))
    pr = tr[tr.purpose_abit.isin(DC.DISC)]
    s.append(KIND_STEP(pr))
    mm = t.merge(tr[["person_id", "start_min", "end_min", "primary_arr", "primary_dep", "purpose_abit", "p_tripid"]], on="person_id")
    mm = mm[(mm.dep_min >= mm.start_min) & (mm.arr_min <= mm.end_min) & (mm.d_purpose != "HOME") & (mm.tripid != mm.p_tripid)]
    pos = np.where(mm.arr_min < mm.primary_arr, "out", np.where(mm.dep_min >= mm.primary_dep, "in", ""))
    st = mm[(pos != "") & mm.d_purpose.isin(DC.DISC)].drop_duplicates("tripid")
    s[-1] = ("discretionary stops before / after the primary activity of these tours", len(st))
    rec = st[st.d_purpose.isin(['RECREATION'])]
    s.append(("records of this model (purpose, primary / stop)", len(rec)))
    rec = rec[rec.person_id.isin(P.index)]; s.append(("person in the survey population", len(rec)))
    sel = raw_rec[h][(raw_rec[h].kind == 'stop') & raw_rec[h].purpose.isin(['RECREATION'])]
    s.append(("every trip end located in the model zones (shipped records)", len(sel)))
    steps[h] = s; N_RAW[h] = len(sel)
reasons = {"tours of the half": "households of the other half",
           "home-based closed tours": "open tours and work-based subtours",
           "primary activity known and not home": "primary activity home or missing",
           "trip to the primary activity found": "no trip arrives at the primary activity time",
           "tours with a shopping, recreation, accompany or other primary activity": "work, school or home primary activity",
           "discretionary stops before / after the primary activity of these tours": "one row per stop (trips to work, school or home are not stops)",
           "records of this model (purpose, primary / stop)": "other purposes",
           "person in the survey population": "person not in the survey population (imputed members)",
           "every trip end located in the model zones (shipped records)": "destination, origin or home not located in a model zone"}
nt.funnel2(steps, reasons)

### 2.2 Rebuild check

`prep_dc_taz.py` (locating the trip ends, writing `dc_rec`, `dc_cand`, `dc_cfg` of both halves), then
`build_estimation_tables.py` and `build_biogeme_data.py` are run in a scratch copy of `input/code` next to this folder
(the scripts write into their own `input/data`; the shipped files are not touched), and every rebuilt table is compared
with the shipped one. About 5-8 min. The script's own final comparison with an older record file is not run (that file is
not shipped).

In [ ]:
# 2.2 Rebuild check
if RUN_DATA_PREP:
    t0 = time.time()
    SCR = nt.scratch_tree(FOLDER, raw_files=[])
    SCODE = os.path.join(SCR, "input", "code")
    nt.run('python3 -c "import prep_dc_taz as T; LOC = T.Locator(); [T.build(h, LOC) for h in (\'estimation\', \'validation\')]"', cwd=SCODE)
    nt.run("python3 build_estimation_tables.py", cwd=SCODE)
    nt.run("python3 build_biogeme_data.py", cwd=SCODE)
    res = pd.concat([nt.compare_trees(os.path.join(SCR, "input", "data", "raw"), RAW, only=r"dc_rec|dc_cand"+ "|dc_cfg"),
                     nt.compare_trees(os.path.join(SCR, "input", "data", "estimation"), os.path.join(FOLDER, "input", "data", "estimation")).assign(half="estimation"),
                     nt.compare_trees(os.path.join(SCR, "input", "data", "holdout"), os.path.join(FOLDER, "input", "data", "holdout")).assign(half="hold-out")],
                    ignore_index=True)
    nt.drop_scratch(SCR)
    print(f"rebuild and comparison: {(time.time() - t0) / 60:.1f} min")
    display(res)
    nt.rebuild_verdict(res, glob.glob(os.path.join(FOLDER, "model", "estimate_*.py"))[0])
else:
    print("skipped (RUN_DATA_PREP = False): the shipped tables are used")

## 3. Choice set and estimation data

*The table the model reads. Look for: the columns used and what they mean (data dictionary).*

The alternatives are all 1,588 model zones (no sampling). The observed location of a record is not one zone: its survey
tract (or traffic zone) is spread over the zones it overlaps, with weights `b` (candidates), and the anchor of the record
(origin of the trip to the stop (anchor a)) can also be spread over several zones with weights `w` (configurations). The likelihood of record n is

  L_n = Σ_c w_c Σ_k b_k exp(V_c(z_k)) / Σ_j exp(V_c(j)),  j = 1 … 1,588.

In [ ]:
# The Biogeme table is wide: one row per record, with the chooser flags f_*, the candidates (cz_k, b_k, zone att
ZONES = pd.read_csv(os.path.join(nt.ABIT_SHARED, "zones", "zones_model_attributes.csv")).sort_values("idx").reset_index(drop=True)
ZONES["area_group"] = nt.area_group_of_county(ZONES.countyfips)

def read_tables(sub):
    base = os.path.join(FOLDER, "input", "data", sub)
    segs = sorted(d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))) or [""]
    out = {}
    for k in ["records", "candidates", "configurations"]:
        out[k] = pd.concat([pd.read_parquet(os.path.join(base, s, f"{k}.parquet")).assign(**({"group": s or "all"} if k == "records" else {})) for s in segs], ignore_index=True)
    return out

EST, HOLD = read_tables("estimation"), read_tables("holdout")
display(pd.DataFrame({h: {k: f"{len(v):,}" for k, v in T.items()} for h, T in [("estimation", EST), ("hold-out", HOLD)]}))
rec, cand, cfg = EST["records"], EST["candidates"], EST["configurations"]
print("records by group:", rec.group.value_counts().to_dict())
fig, ax = nt.panels(3, ncol=3, w=4.2, h=2.8)
nt.bar(cand.groupby("rec").size().value_counts().sort_index(), "candidate zones of the observed location", "records", ax=ax[0])
nt.hist(cand.b, "weight b of a candidate zone", "candidate rows", bins=40, ax=ax[1])
nt.bar(cfg.groupby("rec").size().value_counts().sort_index(), "anchor configurations", "records", ax=ax[2])
plt.tight_layout(); plt.show()
ex = rec.rec[rec.rec.isin(cand.groupby("rec").size().pipe(lambda s: s[s > 1].index))].iloc[0]
print(f"example record {ex}: candidate zones and anchor configurations (zone index 0..1587 -> zone id)")
display(cand[cand.rec == ex].assign(zone_id=lambda d: ZONES.id.values[d.zidx]))
display(cfg[cfg.rec == ex].assign(anchor_zone_id=lambda d: ZONES.id.values[d.home_idx]))

The Biogeme table is wide: one row per record, with the chooser flags `f_*`, the candidates (`cz_k`, `b_k`, zone attributes
`cand_*_k`), the configurations (`w_c`, `za_c`, `zb_c`) and, for every configuration c and zone j, the zone variables
`imp_c_j` (car km or detour km), `cgt_c_j` (car generalised time) and `ls_c_j` (mode-choice logsum) where the model uses them.
It is stored in parts of at most 1,000 records.

In [ ]:
# Choice set and estimation data
est = os.path.join(FOLDER, "input", "data", "estimation")
meta = json.load(open(os.path.join(est, "biogeme_meta.json")))
print({k: v for k, v in meta.items() if k != "parts"})
parts = sorted(glob.glob(os.path.join(est, "biogeme_part*.parquet")))
p0 = pd.read_parquet(parts[0])
stem = pd.Series([re.sub(r"(_\d+)+$", "", c) for c in p0.columns])
groups = stem.groupby(stem, sort=False).size().rename("columns").to_frame()
print(f"{len(parts)} parts; part 0: {p0.shape[0]:,} records x {p0.shape[1]:,} columns")
display(groups.T)
dd = os.path.join(est, "data_dictionary.csv")
pd.read_csv(dd) if os.path.exists(dd) else None

## 4. Descriptive statistics

*What the sample looks like. Look for: plausible shares and maps, and the chosen alternatives against the available ones.*

All shares below are over records (the estimation is unweighted); a record with several candidate zones or configurations
counts with the weights b (normalised per record) and w. Distances are car distances (skim of the 2016 base, km) from the
origin of the trip to the stop (anchor a) to the zones of the observed location.

In [ ]:
# Share of the chosen zones by area of the study region (the survey covers the Baltimore and Washington regions)
SK = DC.skims()["d"]                                   # car km, zone index x zone index

def chosen(T):
    c = T["candidates"].copy(); c["bn"] = c.b / c.groupby("rec").b.transform("sum")
    g = c.merge(T["configurations"][["rec", "home_idx", "w"]], on="rec")
    g["p"] = g.bn * g.w                                # weight of (candidate zone, configuration)
    g["km"] = SK[g.home_idx.to_numpy(), g.zidx.to_numpy()]
    g["area_group"] = pd.Categorical(ZONES.area_group.astype(str).to_numpy()[g.zidx.to_numpy()], categories=nt.AREA_GROUPS, ordered=True)
    return g.merge(T["records"][["rec", "group"]], on="rec")

CH, CHH = chosen(EST), chosen(HOLD)
area = (CH.groupby(["group", "area_group"], observed=False).p.sum().unstack(0))
area = 100 * area / area.sum()
display(area.round(2).rename_axis("area group of the chosen zone"))
fig, ax = plt.subplots(figsize=(7.5, 3.2))
(area.mean(axis=1) if area.shape[1] == 1 else area).plot.barh(ax=ax, color=[nt.OBS, nt.LIGHT, "#6b6b6b", "#c4c4c4"][:area.shape[1]], width=0.8, legend=area.shape[1] > 1)
ax.invert_yaxis(); ax.set_xlabel("share of records (%)"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

Share of the chosen zones by area of the study region (the survey covers the Baltimore and Washington regions).

In [ ]:
# Car distance from the anchor to the chosen zone (b- and w-weighted) and the reported trip distance; the table 
edges = [0, 1, 2, 3, 5, 7.5, 10, 15, 20, 30, 50, 301]
labels = ["0-1", "1-2", "2-3", "3-5", "5-7.5", "7.5-10", "10-15", "15-20", "20-30", "30-50", "50+"]
groups_ = sorted(CH.group.unique())
fig, ax = nt.panels(len(groups_) + 1, ncol=min(3, len(groups_) + 1), w=4.3, h=2.9)
for a, g in zip(ax, groups_):
    x = CH[CH.group == g]
    a.hist(x.km.clip(upper=80), bins=np.arange(0, 81, 2), weights=x.p, color=nt.OBS)
    a.set_xlabel(f"car km to the chosen zone, 80 = 80+{'' if g == 'all' else ' (' + g + ')'}"); a.set_ylabel("records")
r = rec.km_rep.clip(upper=80)
ax[-1].hist(r.dropna(), bins=np.arange(0, 81, 2), color=nt.LIGHT); ax[-1].set_xlabel("reported trip distance, km, 80 = 80+ (all)"); ax[-1].set_ylabel("records")
plt.tight_layout(); plt.show()
band = pd.DataFrame({g: nt.binned(CH[CH.group == g].km, edges, labels, CH[CH.group == g].p) for g in groups_}).round(2)
band.index.name = "car km (band)"
med = np.interp(0.5, (CH.sort_values("km").p.cumsum() / CH.p.sum()).to_numpy(), CH.sort_values("km").km.to_numpy())
print(f"median car distance to the chosen zone {med:.1f} km; {band.iloc[:6].sum().mean():.0f} % of the records within 10 km")
band.T

Car distance from the anchor to the chosen zone (b- and w-weighted) and the reported trip distance; the table gives the shares by distance band.

In [ ]:
# Explanatory variables of the choosers (the model uses them as flags f_* interacted with the impedance)
inc_lab = {0: "<50k", 1: "50-100k", 2: "100k+"}; tod_lab = {0: "AM 6-9", 1: "midday 9-15", 2: "PM 15-19", 3: "evening"}
items = [("hhsize", "household size (6 = 6+)", lambda s: s.clip(upper=6)), ("autos", "cars (4 = 4+)", lambda s: s.clip(upper=4)),
         ("inc_cls", "household income ($/year)", lambda s: s.map(inc_lab)), ("occ", "occupation (1 worker, 2 unemployed, 3 student, 4 retired)", lambda s: s),
         ("tod", "arrival time", lambda s: s.map(tod_lab)), ("position", "stop position (out / in)", lambda s: s.replace("", np.nan)),
         ("tour_purpose", "purpose of the tour", lambda s: s.replace("", np.nan)), ("multi", "tour with stops (1)", lambda s: s)]
items = [i for i in items if i[0] in rec and rec[i[0]].astype(str).replace("", np.nan).nunique() > 1]
fig, ax = nt.panels(len(items) + 1, ncol=3, w=4.3, h=2.8)
for a, (c, lab, f) in zip(ax, items):
    v = f(rec[c]).value_counts()
    v = v.reindex([x for x in (list(inc_lab.values()) if c == "inc_cls" else list(tod_lab.values()) if c == "tod" else sorted(v.index)) if x in v.index])
    nt.bar(100 * v / v.sum(), lab, "share of records (%)", ax=a, rot=30 if c in ("tod", "tour_purpose", "inc_cls") else 0)
nt.hist(rec.age, "age (years)", "records", bins=range(0, 96, 5), ax=ax[-1])
plt.tight_layout(); plt.show()

Explanatory variables of the choosers (the model uses them as flags `f_*` interacted with the impedance).

In [ ]:
# Zone attractions over the 1,588 zones (the attraction variable of this model is marked), and the mean attracti
REF = pd.read_csv(os.path.join(FOLDER, "model", "reference_estimates.csv"))
attr = sorted(set(REF["project_name"].astype(str).str.extract(r"lnx_([a-z]+)")[0].dropna()))
print("attraction variable(s) of this model, ln(1 + x):", attr)
zc = {"ret": "retail jobs", "off": "office jobs", "oth": "other jobs", "stu": "school enrolment", "pop": "persons"}
fig, ax = nt.panels(6, ncol=3, w=4.3, h=2.8)
for a, (c, lab) in zip(ax, list(zc.items())):
    a.hist(np.log1p(ZONES[c]), bins=40, color=nt.OBS)
    a.set_xlabel(f"ln(1 + {lab}) of a zone" + ("  [attraction of this model]" if c in attr else "")); a.set_ylabel("zones")
ax[5].hist(np.log(ZONES.area_km2.clip(lower=0.05)), bins=40, color=nt.OBS); ax[5].set_xlabel("ln(zone area, km²)"); ax[5].set_ylabel("zones")
plt.tight_layout(); plt.show()
zch = CH.groupby("zidx").p.sum()
comp = pd.DataFrame({"all zones (mean)": ZONES[list(zc) + ["area_km2"]].mean(),
                     "chosen zones (record-weighted mean)": ZONES[list(zc) + ["area_km2"]].iloc[zch.index].mul(zch.values, axis=0).sum() / zch.sum()}).round(1)
comp.index = [zc.get(i, "zone area (km²)") for i in comp.index]
comp

Zone attractions over the 1,588 zones (the attraction variable of this model is marked), and the mean attraction and area of the chosen zones against all zones.

In [ ]:
# Chosen and anchor zones of the estimation half (Baltimore-Washington area; white: no record)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.6))
nt.zone_map(pd.Series(zch.values, index=ZONES.id.values[zch.index]), "records (chosen zone, b-weighted)", ax=ax[0], log=True)
home = EST["configurations"].groupby("home_idx").w.sum()
nt.zone_map(pd.Series(home.values, index=ZONES.id.values[home.index]), "records (anchor zone, w-weighted)", ax=ax[1], log=True)
ax[0].set_title("chosen zones", fontsize=10); ax[1].set_title("anchor zones", fontsize=10)
plt.tight_layout(); plt.show()

Chosen and anchor zones of the estimation half (Baltimore-Washington area; white: no record). The survey trip weight `w`
of the records is not used by the estimation:

In [ ]:
# Shares of the chosen areas and distance bands in the two halves; the largest difference is printed above the t
print(f"survey trip weight w of the records: mean {rec.w.mean():.1f}, min {rec.w.min():.1f}, max {rec.w.max():.1f} (not used by the likelihood)")
a = CH.groupby("area_group", observed=False).p.sum(); b = CHH.groupby("area_group", observed=False).p.sum()
t = pd.DataFrame({"estimation half %": 100 * a / a.sum(), "hold-out half %": 100 * b / b.sum()})
db = pd.DataFrame({"estimation half %": nt.binned(CH.km, edges, labels, CH.p), "hold-out half %": nt.binned(CHH.km, edges, labels, CHH.p)})
t = pd.concat({"area group": t, "car km": db}).round(2)
t["difference (points)"] = (t.iloc[:, 0] - t.iloc[:, 1]).round(2)
print(f"records: estimation {EST['records'].shape[0]:,}, hold-out {HOLD['records'].shape[0]:,}; largest difference of a share: {t['difference (points)'].abs().max():.1f} points")
t

Shares of the chosen areas and distance bands in the two halves; the largest difference is printed above the table.

### 4.1 Variables in the data

Each explanatory variable of the final specification as it enters the utility, interactions included (read from the model's own utility functions: the part of the utility that each coefficient multiplies; coefficients that share a label, e.g. the copies of a term for several purposes, are shown in one panel). Here: the zones of the observed location (the candidate zones of the location mixture, by their weights) against all zones of the full choice set; for speed, 1,000 stops drawn at random from the estimation half and 150 zones drawn at random for each as the available alternatives. Scale and size-weight parameters at their estimates.

In [ ]:
# every explanatory variable of the specification: chosen against available zones
import varplots as vp
MV_ = af.module(MODEL_FOLDER); R_ = af.reference(MODEL_FOLDER)
FIX = {p: (1.0 if p.startswith("SCALE_") else v) for p, v in zip(R_.parameter, R_.coef) if p.startswith(("SCALE_", "THETA_"))}
PARTS = sorted(glob.glob(os.path.join(FOLDER, "input", "data", "estimation", "biogeme_part*.parquet")))
t0 = time.time()
VP, NREC = vp.location_from_at(MV_, PARTS, vp.abit_labels(MODEL_FOLDER), FIX, n_rows=1000, n_zones=150)
print(f"{len(VP)} variables; stops drawn: {NREC:,}; {time.time() - t0:.0f} s")
vp.location_figures(VP, what="available", unit='stops')

How to read it: dark grey line = the chosen alternatives, light bars = the available alternatives, as a share of each (%); dashed lines are the two means; 0/1 variables are shown as the share with value 1. A continuous variable that is zero at every alternative of a record cannot affect that record's choice, so such records are left out of its panel (e.g. an interaction with a person group; the share kept is in the panel title). The further apart the two, the more the variable separates the choices.

### Data sources

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

## 5. Model specification

*The utility functions, read from the model script; nothing is re-specified here.*

The folder's script `model/estimate_stop_recreation.py`, imported as it is. Its utility functions:

In [ ]:
# Model specification
sys.path.insert(0, os.path.join(FOLDER, "model")); sys.modules.pop("estimate_stop_recreation", None)   # the model folder's script (input/code may hold a namesake)
import estimate_stop_recreation as M
src = inspect.getsource(M)
print(src[:src.find("def read(")])

In [ ]:
# Biogeme table: one row per record; imp_<c>_<zone>, cgt_<c>_<zone> = zone variables of configuration c for all zones,
# cand_* / b_* = the candidate zones of the observed location and their weights. Only the first two parts are read here.
est = os.path.join(FOLDER, "input", "data", "estimation")
parts = sorted(glob.glob(os.path.join(est, "biogeme_part*.parquet")))
df = pd.concat([pd.read_parquet(p) for p in parts[:2]], ignore_index=True)
person = nt.explanatory(df[[c for c in df.columns if c.startswith("f_")]])
ZONES = pd.read_csv(os.path.join(FOLDER, "..", "00_shared_data", "zones", "zones_model_attributes.csv"))

## 6. Correlation of the explanatory variables

*Correlation of the explanatory variables. Look for: |r| of 0.7 or more and VIF of 10 or more; these make single coefficients less precise.*

|r| ≥ 0.7 and VIF ≥ 10 are listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
# Correlation of the explanatory variables
long = nt.explanatory(nt.location_long(df, ZONES))
print(f"zone variables of {min(300, len(df))} records x {len(ZONES):,} zones = {len(long):,} rows")
Cz = nt.corr_heatmap(long, "Destination variables (impedance, generalised time, zone attributes)")
display(nt.high_pairs(Cz)); display(nt.vif(long))
Cp = nt.corr_heatmap(person, "Person and tour variables of the records"); display(nt.high_pairs(Cp))

## 7. Log-likelihood at the reference coefficients

*Log-likelihood at the article coefficients, without estimation. Look for: the two numbers should be equal (to rounding).*

No estimation: the script's `evaluate("estimation", "full")`, i.e. the log-likelihood of the estimation half at the full-precision coefficients
of `model/reference_estimates.csv`. Target: LL(beta) of that file, -15874.937677242058 (article -15874.9;
N = 5064, K = 20). This model takes about 32 min.

In [ ]:
# Log-likelihood at the reference coefficients
LL_BETA = -15874.937677242058
def evaluate(half):
    here = os.getcwd()
    try:
        return M.evaluate(half, 'full') if "at" in inspect.signature(M.evaluate).parameters else M.evaluate(half)
    finally:
        os.chdir(here)

if RUN_CHECK:
    t0 = time.time(); ll_ref = evaluate("estimation")
    print(f"{(time.time() - t0) / 60:.1f} min")
    if isinstance(ll_ref, (int, float)) and LL_BETA is not None:
        print(f"LL at the reference coefficients {ll_ref:,.4f}; LL(beta) of reference_estimates.csv {LL_BETA:,.4f}; difference {ll_ref - LL_BETA:+.4f}")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

*Biogeme estimation, only with RUN_ESTIMATION. Look for: convergence and the run time.*

The script's `estimate()`: Biogeme with its defaults (`output/biogeme.toml`), robust standard errors. Report in
`output/stop_recreation.html` and `.yaml`. Not run here (`RUN_ESTIMATION = False`); a result in `output/` is read only if it is
newer than `model/reference_estimates.csv`.

In [ ]:
# Biogeme estimation
if RUN_ESTIMATION:
    here = os.getcwd(); t0 = time.time()
    try:
        M.estimate(OUT)
    finally:
        os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s")
ref_time = os.path.getmtime(os.path.join(FOLDER, "model", "reference_estimates.csv"))
y = [f for f in sorted(glob.glob(os.path.join(OUT, "*.yaml")), key=os.path.getmtime) if "eval" not in os.path.basename(f) and os.path.getmtime(f) > ref_time]
results = EstimationResults.from_yaml_file(y[-1]) if y else None
print(f"results read from {os.path.basename(y[-1])}" if y else "no current result in output/: the estimate tables of section 9 are skipped")

## 9. Results

*The estimates and what they mean. Look for: coefficients equal to the article within rounding, and the readings below.*

### Estimates in the article format
Coefficient (stars: *** p < 0.01, ** p < 0.05, * p < 0.10, as in the article), robust s.e., p.

In [ ]:
# Results
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### What the coefficients say

One plain line per term (article estimates of `model/reference_estimates.csv`) and the coefficient plot.

In [ ]:
# plain readings of the coefficients and the coefficient plot (95 % robust intervals, grouped by block)
rows = af.coef_rows(MODEL_FOLDER)
display(rd.abit_readings(rows).style.hide(axis="index").set_properties(**{"text-align": "left"}))
fg.coef_figure(rows); plt.show()

How to read it: a point right of zero makes the alternative more likely; an interval that crosses zero is not significant; colours are the alternatives or purposes a term belongs to.

### Goodness of fit

In [ ]:
# Goodness of fit
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
# Comparison with the article (model/reference_estimates.csv)
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "K": results.number_of_free_parameters, "N": 5064})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
# Correlation of the estimates
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

*Fit on the hold-out records, which the estimation did not use. Look for: a log-likelihood per record close to that of the estimation sample, and the maps.*

Log-likelihood of the hold-out half at the reference coefficients (`evaluate("holdout", "full")`).

In [ ]:
# Hold-out sample
if RUN_HOLDOUT:
    ll_hold = evaluate("holdout")
else:
    print("skipped (RUN_HOLDOUT = False)")

### Hold-out check by area

Observed and model shares of the hold-out records by area. The model share is the sum of the predicted probabilities over all 1,588 zones (computed with this model's utilities; the hold-out log-likelihood recomputed from them equals section 10). Map and tables: `output/holdout_map.png`, `output/holdout_puma.csv`, `output/holdout_county.csv`, `output/holdout_stats.json`.

In [ ]:
# hold-out map (copied into output/) and its reading, numbers from output/holdout_stats.json and output/holdout_county.csv
nt.show_png(os.path.join(OUT, "holdout_map.png"))
st = json.load(open(os.path.join(OUT, "holdout_stats.json"))); HC = pd.read_csv(os.path.join(OUT, "holdout_county.csv"))
top = HC.reindex(HC.diff_pp.abs().sort_values(ascending=False).index).head(3)
nm = rd.county_names(os.path.join(FOLDER, "..", "..", "SILO", "00_shared_data", "zone_system", "zoneSystem.csv"))
display(Markdown(
    "How to read it: (a) observed share of the hold-out records by area (the observed location spread over its candidate zones), (b) model share, same scale, "
    "(c) model minus observed in percentage points: red = model too high, blue = too low.\n\n"
    f"Correlation {st['puma_r']:.2f} by PUMA and {st['county_r']:.2f} by county; mean absolute difference {st['puma_mad']:.2f} pp by PUMA "
    f"({'mapped by county: fewer than 10 records per PUMA' if st['level'] == 'county' else 'mapped by PUMA'}). "
    "Largest county differences: " + ", ".join(f"{nm.get(c, c)} {d:+.1f} pp" for c, d in zip(top.county, top.diff_pp)) + ".\n\n"
    "The hold-out half was also used in the specification search."))

## Glossary

- **MAST**: Maryland Activity Scheduling and Travel model: the activity-based model of the MILES platform, built on the ABIT code base (Activity-Based Incremental Transport model).
- **zone**: A traffic analysis zone of the model (1,588 zones).
- **log-likelihood**: Sum over decision makers of ln(probability of the observed choice). Closer to 0 is better.
- **hold-out**: The 20 % (residential, cars) or half (survey models) of the records not used for estimation. It was also used in the specification search, so its fit is not a fully independent test.
- **robust s.e.**: Standard error from the sandwich formula, valid when the model is not exactly right.
- **VIF**: Variance inflation factor: how much the correlation with other variables inflates a coefficient's variance; 10 or more is high.